# 02 — Structured outputs and validation

This notebook parses raw model responses and validates them before any linguistic evaluation.

Validation is split into:

1. JSON parsing
2. schema validation
3. token alignment
4. inventory checks

In [ ]:
!pip -q install jsonschema

In [ ]:
from pathlib import Path
import json, re
import pandas as pd
from jsonschema import Draft202012Validator

PROJECT_DIR = Path('/content/lrec2026_llm_annotator')
DATA_DIR = PROJECT_DIR / 'data' / 'sample'
SCHEMA_DIR = PROJECT_DIR / 'schemas'
OUTPUT_DIR = PROJECT_DIR / 'outputs'

for d in [OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

schema = json.loads((SCHEMA_DIR / 'pos_lemma_morph_schema.json').read_text(encoding='utf-8'))
validator = Draft202012Validator(schema)

df = pd.read_csv(DATA_DIR / 'toy_sentences.csv')
for col in ['tokens', 'gold_pos', 'gold_lemma', 'gold_features']:
    df[col] = df[col].apply(json.loads)
input_by_id = df.set_index('id').to_dict(orient='index')

In [ ]:
raw_files = [OUTPUT_DIR / 'zero_shot_raw.jsonl', OUTPUT_DIR / 'few_shot_raw.jsonl']
missing = [p for p in raw_files if not p.exists()]
if missing:
    raise FileNotFoundError(f'Missing {missing}. Run 01_prompting_zero_few_shot.ipynb first.')
raw = pd.concat([pd.read_json(p, lines=True) for p in raw_files], ignore_index=True)
raw[['sentence_id','mode','raw_response']].head()

## Parser

The parser should be conservative. It can handle simple Markdown code fences, but it should not silently rewrite the model output into a different annotation.

In [ ]:
def extract_json_text(raw_text):
    text = str(raw_text).strip()
    # Remove a simple fenced code block if the entire response is fenced.
    fenced = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", text, flags=re.DOTALL | re.IGNORECASE)
    if fenced:
        text = fenced.group(1).strip()
    return text

def parse_response(raw_text):
    text = extract_json_text(raw_text)
    try:
        return json.loads(text), None
    except Exception as e:
        return None, f'parse_error: {type(e).__name__}: {e}'

parsed = []
for _, rec in raw.iterrows():
    obj, err = parse_response(rec['raw_response'])
    parsed.append({**rec.to_dict(), 'parsed': obj, 'parse_error': err})
parsed = pd.DataFrame(parsed)
parsed[['sentence_id','mode','parse_error']]

## Validators

A prediction may be valid JSON but still unusable because tokens are missing, surfaces changed, or labels are outside the inventory.

In [ ]:
def validate_prediction(rec):
    errors = []
    obj = rec['parsed']
    sentence_id = rec['sentence_id']

    if obj is None:
        return False, [rec['parse_error']]

    # JSON Schema validation.
    for err in sorted(validator.iter_errors(obj), key=lambda e: list(e.path)):
        path = '.'.join(str(p) for p in err.path)
        errors.append(f'schema_error at {path or "<root>"}: {err.message}')

    # Token alignment validation.
    input_row = input_by_id.get(sentence_id)
    if input_row is None:
        errors.append('unknown_sentence_id')
    else:
        input_tokens = input_row['tokens']
        output_tokens = obj.get('tokens', []) if isinstance(obj, dict) else []
        if len(output_tokens) != len(input_tokens):
            errors.append(f'token_count_mismatch: input={len(input_tokens)} output={len(output_tokens)}')
        else:
            for i, (inp, out_tok) in enumerate(zip(input_tokens, output_tokens)):
                surface = out_tok.get('surface') if isinstance(out_tok, dict) else None
                if surface != inp:
                    errors.append(f'surface_mismatch[{i}]: input={inp!r} output={surface!r}')

    return len(errors) == 0, errors

validated_records = []
for _, rec in parsed.iterrows():
    ok, errors = validate_prediction(rec)
    d = rec.to_dict()
    d['is_valid'] = ok
    d['validation_errors'] = errors
    validated_records.append(d)

validated = pd.DataFrame(validated_records)
validated[['sentence_id','mode','is_valid','validation_errors']]

In [ ]:
valid_path = OUTPUT_DIR / 'validated_predictions.jsonl'
invalid_path = OUTPUT_DIR / 'invalid_outputs.csv'

# JSONL cannot directly serialise nested objects from pandas unless we keep them JSON-compatible.
validated.to_json(valid_path, orient='records', lines=True, force_ascii=False)
validated[~validated['is_valid']][['sentence_id','mode','validation_errors','raw_response']].to_csv(invalid_path, index=False, encoding='utf-8')

print('Wrote', valid_path)
print('Wrote', invalid_path)
validated.groupby(['mode','is_valid']).size().reset_index(name='n')

## Inspect invalid outputs

Invalid outputs are not just technical noise. The invalid-output rate belongs in the method section.

In [ ]:
invalid = validated[~validated['is_valid']]
if len(invalid) == 0:
    print('No invalid outputs.')
else:
    display(invalid[['sentence_id','language','mode','validation_errors','raw_response']])

## Participant TODO

Pick one invalid output. Decide whether the fix should be:

- prompt change;
- schema change;
- preprocessing/tokenisation change;
- expert review;
- no automatic fix, only reporting.

Continue with `03_evaluation_and_error_analysis.ipynb`.